# Relatório do Laboratório 5 — Câmera Estéreo

### ESZA019 — Visão Computacional · UFABC · 2026.2

---

**Autores:**

| # | Nome completo | RA |
|---|---------------|----|
| 1 | *Antonio Carlos de Freitas Vidal Junior* | *11201920894* |
| 2 | *João Vitor De Oliveira Lamano* | *11202022114* |
| 3 | *Willian Kenji Takaracy* | *11201812251* |


## 1. Introdução

Enxergamos o mundo em três dimensões porque temos **dois olhos**: cada um capta a
cena de um ponto de vista ligeiramente diferente, e o cérebro combina as duas
imagens para perceber profundidade. A **visão estéreo** (estereoscopia) reproduz
essa ideia em Visão Computacional usando **duas câmeras** lado a lado — e é a base
de aplicações como reconstrução 3D, medição de distâncias, robótica e o clássico
efeito de **imagem 3D anáglifo** (aquele dos óculos com lentes vermelha e ciano).

Neste quinto laboratório, os objetivos foram entender a **geometria epipolar** que
relaciona duas câmeras, **construir uma câmera estéreo** simples com duas webcams
USB fixadas paralelamente (com a distância entre os eixos ópticos próxima da
distância interpupilar de uma pessoa), **calibrar** esse par estéreo e gerar uma
**imagem/vídeo 3D anáglifo**.

O relatório está organizado assim: uma **fundamentação teórica** respondendo às
questões da Parte 1 (epipolos, plano e linha epipolar, matriz fundamental e
disparidade estéreo); os **procedimentos experimentais**, cobrindo a construção da
câmera (Parte 2) e a execução passo a passo da captura, calibração e geração de
vídeo 3D com as nossas webcams (Parte 3, itens B a D); e a **análise e discussão**,
onde listamos os parâmetros obtidos, avaliamos criticamente a **qualidade da
calibração** e refletimos sobre o uso da câmera estéreo no trabalho final (item E).


## 2. Fundamentação Teórica

### 2.1 Estereoscopia e a ideia de profundidade

Com **duas câmeras** observando a mesma cena a partir de posições diferentes, um
mesmo ponto 3D projeta-se em posições distintas nas duas imagens. Essa diferença de
posição é a chave para recuperar a **profundidade**: quanto mais perto o objeto,
maior a diferença; quanto mais longe, menor. A **geometria epipolar** é o conjunto
de relações que descreve, de forma exata, como os dois pontos de vista se
relacionam — e é o que responderemos a seguir (Parte 1).

### 2.2 Epipolos, plano epipolar e linha epipolar

Considere as duas câmeras com centros ópticos $O_L$ e $O_R$, e um ponto 3D $X$ da
cena. A reta que liga os dois centros é chamada de **linha de base** (*baseline*).

- **Epipolo:** é o ponto onde a linha de base fura o plano de imagem de cada câmera.
  Equivalentemente, o epipolo de uma imagem é a **projeção do centro óptico da outra
  câmera** naquela imagem. Todas as linhas epipolares de uma imagem passam pelo seu
  epipolo.
- **Plano epipolar:** é o plano definido pelo ponto 3D $X$ e pelos dois centros
  ópticos $O_L$ e $O_R$. Para cada ponto da cena existe um plano epipolar (todos
  contêm a linha de base).
- **Linha epipolar:** é a **interseção do plano epipolar com o plano de imagem**.
  Ela traduz a chamada *restrição epipolar*: dado um ponto na imagem da esquerda, o
  seu correspondente na imagem da direita **está necessariamente sobre uma reta** (a
  linha epipolar), e não em qualquer lugar da imagem. Isso reduz a busca por
  correspondências de 2D para 1D — uma enorme simplificação, especialmente após a
  *retificação*, que deixa as linhas epipolares **horizontais**.

### 2.3 A matriz fundamental e seus parâmetros

A geometria epipolar em coordenadas de **pixel** é resumida pela **matriz
fundamental $F$** (3×3), que satisfaz, para todo par de pontos correspondentes
$x$ (esquerda) e $x'$ (direita):

$$
x'^{\mathsf T}\, F\, x = 0
$$

Características (parâmetros) de $F$:

- É uma matriz $3\times 3$, mas de **posto 2** (é singular, $\det F = 0$);
- É definida **a menos de escala**, o que lhe dá **7 graus de liberdade**;
- Os **epipolos** são os vetores nulos de $F$ ($F e = 0$ para o epipolo esquerdo e
  $F^{\mathsf T} e' = 0$ para o direito);
- Ela dá a linha epipolar de um ponto diretamente: $l' = F x$.

Quando se conhecem os parâmetros intrínsecos das câmeras ($K_L$, $K_R$), usa-se a
**matriz essencial** $E = K_R^{\mathsf T} F\, K_L$, que opera em coordenadas
normalizadas, tem **5 graus de liberdade** e se decompõe na rotação $R$ e na
translação $t$ **entre as duas câmeras** — ou seja, na pose relativa do par estéreo.

### 2.4 Disparidade estéreo

Depois de **retificar** o par (alinhar as duas imagens de modo que as linhas
epipolares fiquem horizontais e coincidentes), o correspondente de um ponto aparece
na **mesma linha** nas duas imagens, deslocado apenas na horizontal. Esse
deslocamento é a **disparidade**:

$$
d = x_L - x_R
$$

A disparidade é **inversamente proporcional à profundidade**: pela geometria do par
retificado, $Z = \dfrac{f \cdot B}{d}$, onde $f$ é a distância focal e $B$ é a
*baseline*. Assim, objetos **próximos** têm disparidade **grande** e objetos
**distantes** têm disparidade **pequena** — é exatamente essa relação que permite
transformar um par estéreo em um **mapa de profundidade** e, com os óculos anáglifo,
produzir a sensação de 3D.


## 3. Procedimentos experimentais

### 3.1 Construção da câmera estéreo (Parte 2)

Construímos a câmera estéreo com **duas webcams USB iguais** do laboratório,
fixadas **paralelamente** sobre uma base rígida, seguindo as recomendações da
seção *Steps To Create The Stereo Camera Setup* da referência [2]. Os cuidados
principais foram:

- **Baseline (distância entre os eixos ópticos):** ajustada para ficar próxima da
  **distância interpupilar** medida em um dos integrantes da equipe, imitando o
  espaçamento dos olhos humanos.
- **Paralelismo e rigidez:** as duas câmeras foram alinhadas paralelamente e
  **fixadas firmemente**, pois qualquer movimento relativo entre elas **após a
  calibração** invalida os parâmetros estéreo.

### 3.2 Ambiente e organização (Parte 3)

Os experimentos foram executados em **Linux**, no ambiente **Conda** (`CV26`) com
**OpenCV 4.13**. Partimos dos três códigos do exemplo da referência [2]
(`capture_images.py`, `calibrate.py`, `movie3d.py`), adaptados apenas no necessário
para usar as **webcams ao vivo**. A célula de *setup* abaixo importa as bibliotecas,
define os índices das câmeras (esquerda e direita) e cria as pastas de saída.

In [1]:
import numpy as np
import cv2
import time
import glob
import os
from tqdm import tqdm

# Índices das webcams (câmera ESQUERDA e DIREITA)
CamL_id = 0
CamR_id = 2

# Pastas de saída
pathL = "/home/ufabc/Downloads/CV_LAB_ANTONIO_VIDAL/lab_5_arquivos/data/stereoL/"
pathR = "/home/ufabc/Downloads/CV_LAB_ANTONIO_VIDAL/lab_5_arquivos/data/stereoR/"
os.makedirs(pathL, exist_ok=True)
os.makedirs(pathR, exist_ok=True)

print("OpenCV:", cv2.__version__)
print("Pastas prontas:", pathL, pathR)


OpenCV: 4.13.0
Pastas prontas: /home/ufabc/Downloads/CV_LAB_ANTONIO_VIDAL/lab_5_arquivos/data/stereoL/ /home/ufabc/Downloads/CV_LAB_ANTONIO_VIDAL/lab_5_arquivos/data/stereoR/


**Verificação rápida da atribuição L/R.** Antes de capturar, este trecho de
*preview* abre as duas câmeras e mostra cada uma em sua janela, para confirmar qual
webcam é a esquerda e qual é a direita (evitando trocar os lados na calibração).

```python
# Preview rápido para checar a atribuição L/R
CamL = cv2.VideoCapture(CamL_id)
CamR = cv2.VideoCapture(CamR_id)
for _ in range(30):
    CamL.read(); CamR.read()
retL, frameL = CamL.read()
retR, frameR = CamR.read()
CamL.release(); CamR.release()
if retL and retR:
    cv2.imshow('Esquerda (L)', frameL)
    cv2.imshow('Direita (R)', frameR)
    cv2.waitKey(0)
    cv2.destroyAllWindows()
else:
    print("Não li as câmeras. Ajuste CamL_id/CamR_id (tente 0,2,4... no Linux).")
```

No Linux, os índices `/dev/videoN` nem sempre são sequenciais, então a célula
abaixo **testa quais índices realmente abrem e leem quadro**, para escolher os
corretos para as câmeras esquerda e direita.

In [2]:
import cv2
disponiveis = []
for i in range(10):
    cap = cv2.VideoCapture(i, cv2.CAP_V4L2)
    ok = cap.isOpened()
    ret = False
    if ok:
        ret, _ = cap.read()
    cap.release()
    if ok and ret:
        disponiveis.append(i)
    print(f"index {i:>2}: abre={ok}  le_frame={ret}")
print("\n>>> Índices que funcionam de verdade:", disponiveis)

index  0: abre=True  le_frame=True
index  1: abre=False  le_frame=False


index  2: abre=True  le_frame=True
index  3: abre=False  le_frame=False
index  4: abre=False  le_frame=False
index  5: abre=False  le_frame=False
index  6: abre=False  le_frame=False
index  7: abre=False  le_frame=False
index  8: abre=False  le_frame=False
index  9: abre=False  le_frame=False

>>> Índices que funcionam de verdade: [0, 2]


### 3.3 Item (C) — Captura das imagens de calibração

## 2. Capturar imagens — `capture_images.py`

Versão do `capture_images.py`. Adaptação: os índices vêm do Setup e as pastas já existem
(removi a etapa interativa de troca de IDs, que usava o preview acima). A cada `T` segundos,
se o tabuleiro (9×6) for detectado nas duas câmeras, o par é salvo. **ESC** encerra.

A célula a seguir abre as duas webcams simultaneamente, detecta o tabuleiro em
tempo real nas duas imagens e, quando o padrão aparece **nas duas** ('DETECTADO'),
salva o par ao apertar **`s`** (ou espaço); **ESC** encerra. As fotos salvas são as
originais (sem os desenhos de sobreposição).

In [3]:
CamL = cv2.VideoCapture(0)
CamR = cv2.VideoCapture(2)
output_path = "/home/ufabc/Downloads/CV_LAB_ANTONIO_VIDAL/lab_5_arquivos/data/"
count = 0

# Flags que deixam a detecção ao vivo mais rápida/robusta
find_flags = (cv2.CALIB_CB_ADAPTIVE_THRESH +
              cv2.CALIB_CB_NORMALIZE_IMAGE +
              cv2.CALIB_CB_FAST_CHECK)

print("Instruções:")
print("  - posicione o tabuleiro (9x6) visível NAS DUAS câmeras")
print("  - aperte 's' (ou ESPACO) para SALVAR o par quando aparecer 'DETECTADO'")
print("  - aperte ESC para encerrar")

while True:
    retL, frameL = CamL.read()
    retR, frameR = CamR.read()
    if not (retL and retR):
        print("Falha ao ler as câmeras."); break

    grayL = cv2.cvtColor(frameL, cv2.COLOR_BGR2GRAY)
    grayR = cv2.cvtColor(frameR, cv2.COLOR_BGR2GRAY)

    okL, cornersL = cv2.findChessboardCorners(grayL, (6, 8), find_flags)
    okR, cornersR = cv2.findChessboardCorners(grayR, (6, 8), find_flags)

    # Imagens só para exibição (as fotos salvas são as originais, sem desenhos)
    dispL, dispR = frameL.copy(), frameR.copy()
    if okL: cv2.drawChessboardCorners(dispL, (6, 8), cornersL, okL)
    if okR: cv2.drawChessboardCorners(dispR, (6, 8), cornersR, okR)

    detectado = okL and okR
    cor = (0, 200, 0) if detectado else (0, 0, 200)
    txt = "DETECTADO - aperte 's' para salvar" if detectado else "procurando tabuleiro..."
    cv2.putText(dispL, txt, (20, 40), cv2.FONT_HERSHEY_SIMPLEX, 0.8, cor, 2)
    cv2.putText(dispL, "salvos: %d" % count, (20, 75), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (255, 0, 0), 2)

    cv2.imshow('imgL', dispL)
    cv2.imshow('imgR', dispR)

    key = cv2.waitKey(1) & 0xFF
    if key == 27:                         # ESC
        break
    elif key == ord('s') or key == 32:    # 's' ou ESPACO
        if detectado:
            count += 1
            cv2.imwrite(output_path + 'stereoL/img%d.png' % count, frameL)
            cv2.imwrite(output_path + 'stereoR/img%d.png' % count, frameR)
            print("  par %d salvo" % count)
        else:
            print("  tabuleiro nao detectado nas duas cameras - reposicione")

CamL.release()
CamR.release()
cv2.destroyAllWindows()
print("Total de pares salvos:", count)

Instruções:
  - posicione o tabuleiro (9x6) visível NAS DUAS câmeras
  - aperte 's' (ou ESPACO) para SALVAR o par quando aparecer 'DETECTADO'
  - aperte ESC para encerrar


  par 1 salvo
  par 2 salvo
  par 3 salvo
  par 4 salvo
  tabuleiro nao detectado nas duas cameras - reposicione
  par 5 salvo
  par 6 salvo
  par 7 salvo
  par 8 salvo
  par 9 salvo
  par 10 salvo
  par 11 salvo
  par 12 salvo
  tabuleiro nao detectado nas duas cameras - reposicione
  par 13 salvo
  tabuleiro nao detectado nas duas cameras - reposicione
  par 14 salvo
  par 15 salvo
  par 16 salvo
  par 17 salvo
  par 18 salvo
  tabuleiro nao detectado nas duas cameras - reposicione
  par 19 salvo
  par 20 salvo
Total de pares salvos: 20


### 3.4 Item (C) — Calibração da câmera estéreo

## 3. Calibrar — `calibrate.py`

O `calibrate.py` original. Única adaptação necessária: em vez do `range(1,28)` fixo (27 imagens),
o laço percorre **quantas imagens você capturou** (`img1.png … imgN.png`). O resto é idêntico ao
exemplo. Ao final, salva `data/params_py.xml`. Aperte uma tecla na janela para avançar cada par.

In [4]:
print("Extracting image coordinates of respective 3D pattern ....\n")

criteria = (cv2.TERM_CRITERIA_EPS + cv2.TERM_CRITERIA_MAX_ITER, 30, 0.001)

objp = np.zeros((6*8, 3), np.float32)
objp[:, :2] = np.mgrid[0:6, 0:8].T.reshape(-1, 2)

img_ptsL = []
img_ptsR = []
obj_pts = []

# Adaptação: número de imagens = quantas foram capturadas (antes era range(1,28) fixo)
n_imgs = len(glob.glob(pathL + "img*.png"))
print("Imagens de calibração encontradas:", n_imgs)

for i in tqdm(range(1, n_imgs + 1)):
    imgL = cv2.imread(pathL + "img%d.png" % i)
    imgR = cv2.imread(pathR + "img%d.png" % i)
    imgL_gray = cv2.imread(pathL + "img%d.png" % i, 0)
    imgR_gray = cv2.imread(pathR + "img%d.png" % i, 0)

    outputL = imgL.copy()
    outputR = imgR.copy()

    retR, cornersR = cv2.findChessboardCorners(outputR, (6, 8), None)
    retL, cornersL = cv2.findChessboardCorners(outputL, (6, 8), None)

    if retR and retL:
        obj_pts.append(objp)
        cv2.cornerSubPix(imgR_gray, cornersR, (11, 11), (-1, -1), criteria)
        cv2.cornerSubPix(imgL_gray, cornersL, (11, 11), (-1, -1), criteria)
        cv2.drawChessboardCorners(outputR, (6, 8), cornersR, retR)
        cv2.drawChessboardCorners(outputL, (6, 8), cornersL, retL)
        cv2.imshow('cornersR', outputR)
        cv2.imshow('cornersL', outputL)
        cv2.waitKey(0)

        img_ptsL.append(cornersL)
        img_ptsR.append(cornersR)

cv2.destroyAllWindows()

print("Calculating left camera parameters ... ")
retL, mtxL, distL, rvecsL, tvecsL = cv2.calibrateCamera(obj_pts, img_ptsL, imgL_gray.shape[::-1], None, None)
hL, wL = imgL_gray.shape[:2]
new_mtxL, roiL = cv2.getOptimalNewCameraMatrix(mtxL, distL, (wL, hL), 1, (wL, hL))

print("Calculating right camera parameters ... ")
retR, mtxR, distR, rvecsR, tvecsR = cv2.calibrateCamera(obj_pts, img_ptsR, imgR_gray.shape[::-1], None, None)
hR, wR = imgR_gray.shape[:2]
new_mtxR, roiR = cv2.getOptimalNewCameraMatrix(mtxR, distR, (wR, hR), 1, (wR, hR))

print("Stereo calibration .....")
flags = 0
flags |= cv2.CALIB_FIX_INTRINSIC
criteria_stereo = (cv2.TERM_CRITERIA_EPS + cv2.TERM_CRITERIA_MAX_ITER, 30, 0.001)

retS, new_mtxL, distL, new_mtxR, distR, Rot, Trns, Emat, Fmat = cv2.stereoCalibrate(
    obj_pts, img_ptsL, img_ptsR, new_mtxL, distL, new_mtxR, distR,
    imgL_gray.shape[::-1], criteria_stereo, flags)

rectify_scale = 1  # 0 imagem cortada, 1 imagem não cortada
rect_l, rect_r, proj_mat_l, proj_mat_r, Q, roiL, roiR = cv2.stereoRectify(
    new_mtxL, distL, new_mtxR, distR, imgL_gray.shape[::-1], Rot, Trns, rectify_scale, (0, 0))

Left_Stereo_Map = cv2.initUndistortRectifyMap(
    new_mtxL, distL, rect_l, proj_mat_l, imgL_gray.shape[::-1], cv2.CV_16SC2)
Right_Stereo_Map = cv2.initUndistortRectifyMap(
    new_mtxR, distR, rect_r, proj_mat_r, imgR_gray.shape[::-1], cv2.CV_16SC2)

print("Saving parameters ......")
cv_file = cv2.FileStorage("data/params_py.xml", cv2.FILE_STORAGE_WRITE)
cv_file.write("Left_Stereo_Map_x", Left_Stereo_Map[0])
cv_file.write("Left_Stereo_Map_y", Left_Stereo_Map[1])
cv_file.write("Right_Stereo_Map_x", Right_Stereo_Map[0])
cv_file.write("Right_Stereo_Map_y", Right_Stereo_Map[1])
cv_file.release()
print("OK.")


Extracting image coordinates of respective 3D pattern ....

Imagens de calibração encontradas: 20


Calculating left camera parameters ... 
Calculating right camera parameters ... 
Stereo calibration .....
Saving parameters ......
OK.


**Resultados e parâmetros da calibração (item C).** Foram capturados **20 pares**
de imagens; destes, o tabuleiro (6×8) foi detectado nas **duas** câmeras em **11
pares**, que foram os efetivamente usados na calibração estéreo. O pipeline do
`calibrate.py` calcula, em sequência:

1. **Intrínsecos de cada câmera** — `cv2.calibrateCamera` para a esquerda e para a
   direita, gerando as matrizes $K_L$, $K_R$ e os vetores de distorção $dist_L$,
   $dist_R$ (refinados com `getOptimalNewCameraMatrix`);
2. **Calibração estéreo** — `cv2.stereoCalibrate` (com `CALIB_FIX_INTRINSIC`),
   que estima a **pose relativa** entre as câmeras: a matriz de **rotação $R$**, o
   vetor de **translação $T$** (a *baseline*), a **matriz essencial $E$** e a
   **matriz fundamental $F$**, além do erro de reprojeção **RMS**;
3. **Retificação** — `cv2.stereoRectify` produz as transformações $R_1, R_2$, as
   matrizes de projeção $P_1, P_2$ e a matriz de reprojeção $Q$ (que converte
   disparidade em profundidade);
4. **Mapas de retificação** — `cv2.initUndistortRectifyMap` gera os mapas que
   alinham as duas imagens; são eles que o `movie3d.py` usa depois.

Os valores numéricos disponíveis na saída foram:

$$
\text{RMS estéreo} = 190{,}85 \qquad
T \approx \begin{bmatrix} 41{,}81 \\ 12{,}77 \\ -135{,}06 \end{bmatrix}\ \text{(em unidades de quadrado do tabuleiro)}
$$

**Quais parâmetros foram salvos no arquivo XML?** No código deste exemplo, o arquivo
`data/params_py.xml` guarda **apenas os quatro mapas de retificação** —
`Left_Stereo_Map_x`, `Left_Stereo_Map_y`, `Right_Stereo_Map_x` e
`Right_Stereo_Map_y`. Esses mapas já **embutem** os intrínsecos, a distorção e as
transformações de retificação de cada câmera (por isso o `movie3d.py` precisa só
deles para alinhar as imagens ao vivo). As matrizes $K_L$, $K_R$, $dist$, $R$, $T$,
$E$, $F$ e $Q$ são **calculadas mas não gravadas** no XML por este código — se
quiséssemos reutilizá-las, seria preciso adicioná-las explicitamente ao
`FileStorage`.


### 3.5 Itens (B) e (D) — Imagem/vídeo 3D anáglifo

O `movie3d.py` lê o `params_py.xml`, aplica os mapas de retificação às duas câmeras
ao vivo com `cv2.remap`, combina a imagem **esquerda no canal vermelho** e a
**direita nos canais verde/azul (ciano)** e exibe o resultado anáglifo — que é visto
em 3D com os óculos de lentes **vermelho/ciano**. A célula abaixo é a versão
adaptada para o item (D): ela **grava** ~15 s do anáglifo em `data/anaglifo.avi`.
O arquivo necessário para rodar é justamente o `params_py.xml` da calibração.

In [6]:
# ---- Gravação de vídeo 3D anáglifo (~10-20s) ----
DURACAO_S = 15          # duração da gravação
FPS       = 20          # quadros por segundo do arquivo
TAMANHO   = (700, 700)  # resolução do vídeo salvo
SAIDA_AVI = "data/anaglifo.avi"
SAIDA_MP4 = "data/anaglifo.mp4"

CamL = cv2.VideoCapture(CamL_id, cv2.CAP_V4L2)
CamR = cv2.VideoCapture(CamR_id, cv2.CAP_V4L2)
for _ in range(30):     # aquecimento
    CamL.read(); CamR.read()

cv_file = cv2.FileStorage("data/params_py.xml", cv2.FILE_STORAGE_READ)
Left_Stereo_Map_x  = cv_file.getNode("Left_Stereo_Map_x").mat()
Left_Stereo_Map_y  = cv_file.getNode("Left_Stereo_Map_y").mat()
Right_Stereo_Map_x = cv_file.getNode("Right_Stereo_Map_x").mat()
Right_Stereo_Map_y = cv_file.getNode("Right_Stereo_Map_y").mat()
cv_file.release()

# Objeto que grava o vídeo (XVID = .avi bem compatível)
fourcc = cv2.VideoWriter_fourcc(*"XVID")
writer = cv2.VideoWriter(SAIDA_AVI, fourcc, FPS, TAMANHO)

print("Gravando ~%ds ... (ESC para parar antes)" % DURACAO_S)
inicio = time.time()
try:
    while time.time() - inicio < DURACAO_S:
        retR, imgR = CamR.read()
        retL, imgL = CamL.read()
        if not (retL and retR):
            break

        Left_nice  = cv2.remap(imgL, Left_Stereo_Map_x, Left_Stereo_Map_y,
                               cv2.INTER_LANCZOS4, cv2.BORDER_CONSTANT, 0)
        Right_nice = cv2.remap(imgR, Right_Stereo_Map_x, Right_Stereo_Map_y,
                               cv2.INTER_LANCZOS4, cv2.BORDER_CONSTANT, 0)

        output = Right_nice.copy()
        output[:, :, 2] = Left_nice[:, :, 2]     # vermelho = esquerda; ciano = direita
        output = cv2.resize(output, TAMANHO)

        writer.write(output)                     # <-- grava o quadro no arquivo
        cv2.imshow("Gravando 3D", output)
        if cv2.waitKey(1) & 0xFF == 27:
            break
finally:
    CamL.release()
    CamR.release()
    writer.release()                             # <-- fecha o arquivo (importante!)
    cv2.destroyAllWindows()

print("Vídeo salvo em:", SAIDA_AVI)

# ---- Converte para mp4 (o roteiro pede o formato mp4) ----
ok = os.system('ffmpeg -y -i "%s" -c:v libx264 -pix_fmt yuv420p "%s"' % (SAIDA_AVI, SAIDA_MP4))
if ok == 0:
    print("Convertido para:", SAIDA_MP4)
else:
    print("ffmpeg indisponível. Alternativa: grave direto em mp4 trocando o fourcc "
          "por *'mp4v' e SAIDA por 'data/anaglifo.mp4'.")

Gravando ~15s ... (ESC para parar antes)
Vídeo salvo em: data/anaglifo.avi
ffmpeg indisponível. Alternativa: grave direto em mp4 trocando o fourcc por *'mp4v' e SAIDA por 'data/anaglifo.mp4'.


sh: 1: ffmpeg: not found


In [5]:
CamL.set(cv2.CAP_PROP_FRAME_WIDTH, 640); CamL.set(cv2.CAP_PROP_FRAME_HEIGHT, 480)
CamR.set(cv2.CAP_PROP_FRAME_WIDTH, 640); CamR.set(cv2.CAP_PROP_FRAME_HEIGHT, 480)

False

A célula a seguir foi usada para **conferir a qualidade** da calibração: ela lê os
mapas de retificação do XML e imprime seus limites, além do RMS estéreo, do vetor de
translação e do número de pares usados.

In [7]:
cv_file = cv2.FileStorage("data/params_py.xml", cv2.FILE_STORAGE_READ)
Lx = cv_file.getNode("Left_Stereo_Map_x").mat()
cv_file.release()
print("Lx dtype/shape:", Lx.dtype, Lx.shape)
print("Lx canal 0 (x-origem) min/max:", int(Lx[:,:,0].min()), int(Lx[:,:,0].max()))
print("Lx canal 1 (y-origem) min/max:", int(Lx[:,:,1].min()), int(Lx[:,:,1].max()))
print("  (esperado ~ 0..640 e 0..480; se estiver tipo -32000/32000, calibração ruim)\n")

# Qualidade da calibração (se as variáveis existirem do item 3)
try:
    print("RMS estéreo (retS):", retS, "  <- bom é < ~1.0; ruim é dezenas/centenas")
    print("Translação T (baseline, em unidades de quadrado do tabuleiro):\n", Trns.ravel())
    print("Nº de pares usados na calibração:", len(obj_pts))
except NameError:
    print("Rode o item 3 neste kernel para ver retS/Trns/obj_pts.")

Lx dtype/shape: int16 (480, 640, 2)
Lx canal 0 (x-origem) min/max: 10868 19249
Lx canal 1 (y-origem) min/max: -16539 -10067
  (esperado ~ 0..640 e 0..480; se estiver tipo -32000/32000, calibração ruim)

RMS estéreo (retS): 190.8528049699323   <- bom é < ~1.0; ruim é dezenas/centenas
Translação T (baseline, em unidades de quadrado do tabuleiro):
 [  41.81396575   12.77432694 -135.06157237]
Nº de pares usados na calibração: 11


> **Vídeo gerado:** o anáglifo foi salvo em **`data/anaglifo.avi`**. A conversão
> para `.mp4` (pedida no item D) não pôde ser feita na máquina por ausência do
> `ffmpeg`; a alternativa é gravar direto em `.mp4` trocando o *fourcc* por `mp4v`,
> ou converter o `.avi` em outra máquina. O arquivo de vídeo acompanha o relatório
> na pasta da equipe (a ser publicada no GitHub).

## 4. Análise e discussão

### 4.1 Qualidade da calibração estéreo

O ponto mais importante desta análise é o **erro de reprojeção RMS = 190,85**. Como
referência, uma boa calibração estéreo costuma ficar **abaixo de ~1,0 pixel**; um
valor na casa das **centenas** indica que a calibração **não ficou boa**. Três
evidências confirmam isso:

- **RMS altíssimo** (190,85 ≫ 1);
- O **vetor de translação** $T \approx (41{,}8,\ 12{,}8,\ -135{,}1)$ é dominado pela
  componente **$Z$** (profundidade), quando, para duas câmeras lado a lado, ele
  deveria ser dominado pela componente **horizontal ($X$)**, correspondente à
  *baseline*. Ou seja, a geometria estimada está fisicamente incoerente;
- Os **mapas de retificação** têm valores na faixa de ~10 000 a ~19 000 (e negativos
  de ~-16 000), muito fora do esperado ($0..640$ em $x$ e $0..480$ em $y$) — sinal
  claro de uma retificação inválida.

**Prováveis causas.** A causa mais provável, típica em calibração estéreo, é a
**ordem inconsistente dos cantos** detectados entre as imagens esquerda e direita:
com um tabuleiro assimétrico, o `findChessboardCorners` pode retornar os cantos
começando de extremidades opostas em cada câmera, fazendo com que o canto $i$ da
esquerda **não corresponda** ao canto $i$ da direita — o que destrói a estimativa do
`stereoCalibrate`. Contribuem ainda: o número **relativamente baixo de pares válidos
(11)**, possivelmente com **pouca variedade** de distâncias e ângulos do tabuleiro, e
a qualidade dos intrínsecos individuais (fixados via `CALIB_FIX_INTRINSIC`). Vale
notar que a captura usou *flags* de detecção mais robustas
(`ADAPTIVE_THRESH + NORMALIZE_IMAGE + FAST_CHECK`), enquanto a calibração detectou os
cantos sem *flags* — o que ajuda a explicar por que só 11 dos 20 pares entraram.

**Melhorias sugeridas.** Para uma próxima tentativa: (i) **garantir a mesma ordenação
dos cantos** nas duas imagens (ordenar os cantos ou usar
`findChessboardCornersSB`); (ii) capturar **mais pares (15–25)** cobrindo bem o campo
de visão, com o tabuleiro em várias **distâncias, inclinações e posições**; (iii)
melhorar a iluminação e a nitidez; (iv) confirmar o **tamanho físico do quadrado** e
a dimensão correta do tabuleiro; e (v) reforçar a **fixação rígida** das câmeras.

### 4.2 Percepção 3D — ao vivo × gravado (item D)

Com os óculos anáglifo vermelho/ciano, a percepção de profundidade depende
diretamente da qualidade da retificação: como a calibração acima ficou ruim, o efeito
3D tende a vir acompanhado de **desalinhamento vertical** entre os canais, o que
reduz a sensação de profundidade e pode causar **desconforto visual**. Registramos a
percepção individual dos integrantes (a preencher com a observação de cada um):

- **Antonio:** *[percepção individual — ex.: sensação de profundidade fraca, leve
  desconforto pelo desalinhamento]*
- **João:** *[percepção individual]*
- **Willian:** *[percepção individual]*

Sobre **ao vivo × gravado**: na versão **ao vivo** o paralaxe responde ao movimento
em tempo real, mas pode apresentar pequenos **travamentos/dessincronia** entre as
câmeras; na versão **gravada**, o vídeo tem *frame rate* fixo e passa por
**compressão**, o que estabiliza a reprodução, mas pode **suavizar detalhes** e não
permite reagir ao movimento do observador. Em ambos os casos, a base para um bom 3D
continua sendo uma **calibração correta**.

### 4.3 Aplicação no Trabalho Final (item E)

Uma câmera estéreo é, essencialmente, um **sensor de profundidade de baixo custo**.
No trabalho final, ela pode ser usada para tarefas que exigem medir distâncias ou
reconstruir a geometria da cena: **detecção de obstáculos e navegação** de um robô
móvel, **medição de dimensões** de peças e **reconstrução 3D** de objetos. Conectando
com a área de automação e robótica, um uso concreto seria a **inspeção
tridimensional em linha de produção** — verificar o volume/altura de peças ou detectar
falhas de montagem que uma câmera 2D não capturaria —, aproveitando o mapa de
disparidade/profundidade que o par estéreo fornece. Em todos esses casos, os
conceitos deste laboratório (geometria epipolar, calibração estéreo e disparidade)
são o núcleo do sistema, e a lição sobre **qualidade de calibração** é decisiva para
que as medidas sejam confiáveis.


## 5. Conclusões

Neste laboratório entendemos os fundamentos da **estereoscopia** e da **geometria
epipolar** — epipolos, plano e linha epipolar, a matriz fundamental e a disparidade —
e construímos uma **câmera estéreo** com duas webcams, com *baseline* próxima da
distância interpupilar, para realizar um experimento de imagem 3D anáglifo.

Na prática, capturamos os pares de calibração, executamos a **calibração estéreo**
(que estima $K$, distorção, $R$, $T$, $E$, $F$ e gera os mapas de retificação salvos
no `params_py.xml`) e geramos o **vídeo 3D** anáglifo. Um resultado central — e
honesto — foi que a calibração ficou com **RMS ≈ 190**, muito acima do ideal,
evidenciando uma calibração de baixa qualidade; analisamos as prováveis causas
(ordenação inconsistente dos cantos, poucos pares e pouca variedade de poses) e
propusemos melhorias concretas.

Mais do que um mosaico "perfeito", o laboratório consolidou o entendimento de **por
que** a calibração estéreo é sensível e de como cada parâmetro (intrínsecos,
extrínsecos, retificação) participa da geração de profundidade — uma base essencial
para aplicações reais de visão 3D, como robótica e inspeção industrial.


## 6. Referências

1. LearnOpenCV — *Introduction to Epipolar Geometry and Stereo Vision*.
   https://learnopencv.com/introduction-to-epipolar-geometry-and-stereo-vision/
2. LearnOpenCV — *Making A Low-Cost Stereo Camera Using OpenCV*.
   https://learnopencv.com/making-a-low-cost-stereo-camera-using-opencv/
   (código: https://github.com/spmallick/learnopencv/tree/master/stereo-camera)
3. LearnOpenCV — *Understanding Lens Distortion*.
   https://learnopencv.com/understanding-lens-distortion/
4. LOOP, C.; ZHANG, Z. *Computing Rectifying Homographies for Stereo Vision*. IEEE
   CVPR, 1999.
5. LearnOpenCV — *Geometry of Image Formation*.
   https://learnopencv.com/geometry-of-image-formation/
6. HARTLEY, R.; ZISSERMAN, A. *Multiple View Geometry in Computer Vision*. 2ª ed.
   Cambridge University Press, 2004.
7. Roteiro do Laboratório 5 — ESZA019 Visão Computacional, UFABC, 2026.2 (v.2).
